## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


*We will archieve reproducibility, due to the pipelines. As, if correctly leveraged, the piplines themselves document the correct, ordered processing steps taken. So anyone who wants to reproduce the approach just needs the data and the pipelines. As everytime the pipeline is run it should be executed the exact same way.*

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

*The next steps would involve performing quality control (QC) of the files and then the processing of the reads for alignments and quantification in order to obtain a count table. We would use the rnaseq pipeline. This pipeline includes quality assessment, adapter/quality trimming if necessary, removal of contaminants, alignment to the reference genome, and generation of gene-level counts.*

# Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

*The rnaseq pipeline processes RNA sequencing data. It consists of 5 blocks: Pre-processing, Genome alignment & quantification, Pseudo-alignment & quantification, Post-processing, Quality control & reporting. <br>
The main tools in the prost-processing block include FastQC, Trim Galore!, UMI-tools, BBSplit, SortMeRNA, etc. The tools of this step read the quality control, extract UMI, and clean the data.  <br>
The important tools during the Genome alignment & quantification step include an alignment or pseudo-alignment tool such as STAR, Hisat2 or Bowtie2, as well as an quantification tools such as Salmon. Also important are tools such as Picards, which can mark duplicates.* <br>


As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [1]:
import pandas as pd

sample_sheet = pd.read_csv('/home/nina/computational-workflows-2026/day_02/samplesheet.csv')
sample_sheet.rename(columns={'instrument_platform':'seq_platform'}, inplace=True)
sample_sheet = sample_sheet[['sample','fastq_1','fastq_2','seq_platform']]
sample_sheet['strandedness'] = 'auto'

sample_sheet.to_csv('input_sample_sheet.csv',  index=False)

In [ ]:
# post here the command you used to run nf-core/rnaseq
!nextflow run nf-core/rnaseq \
    -c nextflow.config \
    -r 3.27.0 \
    --input input_sample_sheet.csv \
    --outdir rnaseq_output \
    --gtf Mus_musculus.GRCm38.102.gtf \
    --fasta Mus_musculus.GRCm38.dna.chromosome.19.fa \
    --aligner star_salmon \
    -params-file params.yml \
    -profile docker

Explain all the parameters you set and why you set them in this way.



*First of all I added my configuration file to adjust the memory use to my laptops optimum. I also provided the gtf annotation file of Mus Musculus and the fasta file, both downloaded from Ensemble. 
I set my aligner to star_salmon. The aligner parameter specifies what alignment algorithm is used. I decided to use star_salmon, which uses the STAR algorithm. This algorithm works well for rna_seq data, the downside is its rather high memory usage. I decided to try it anyway, as my laptop is responsible for the quantification step.I also provided an additional params-file, this file was added, as unfortunatly the pipline otherwise wrongly interprets the input as a String. In the params.yml I defined the parameter min_mapped-reads as 0, so that samples would not be excluded based on the minimum mapping theshold. This was relevant because only chromosom 19 was used for the mapping due to time reasons.  So a lot of reads might not map well, but we still want to consider all reads. I also set to profile to docker, as the pipeline should be run using docker containers to handle dependencies by the pipeline.* 

## Browsing the results

How did the pipeline perform?

*Unfortunatly the star_salmon pipeline did not manage to complete the alignment step (I was too positive in the answer above), even while running overnight. So I will be using the uploaded files from here on. Looking at the Nextflow workflow report the pipeline took about 1 h 44m 57s, with about 103.8 CPU-Hours. The greatest amount of CPU Usage and Memory was allocated to the Star Aligner Tool. Which seems reasonable, as this tool uses a very computation heavy approach. Salmon and SAMtools also made up a lot of the CPU usage. And the PICARD tool came second in Memory usage. STAR also made up a lot of the Job duration time. 
All these things indicate, that the pipeline can perform rather well, as long as the Device used is able to allocate a lot of memory and CPU usage to the task. Otherwise the usage of the Star aligner should be avoided, as this takes up quite a lot of time. Instead it would make more sense to use Salmon Pseudoaligner for the quantification and Hisat2 for the alignment, as those Tools need much less Computation power and time*. 

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

*The Quality Control consists of multiple steps, that are summarized within the multiqc report. The report begins with general statistics, which include things such as read counts, mapping rates, duplication rate, and GC content. All of these metrics provide an overview of the dataset and help to quickly identify outliers. The pipeline also infers library strandedness. This is important to accuratly asses gene-expression quantification. FastQC, which is listed after the strandedness inference within the report, evaluates the raw and trimmed reads for characterisics such as per-base quality scores, sequence length, GC content, adapter contamination, etc.. The pipeline uses Trim Galore to remove adapter sequences and trim low-quality bases. After alignment STAR and SAMtools provide mapping and alignment statistics. RSeQC and Qualimap evaluate alignment quality and RNA-seq characteristics, while Picard identifies and marks duplicate reads. DupRadar examines duplication in relation to gene expression, which allows for the distinction of expected duplication vs technical duplication. The pipeline also includes some analysis useing DESeq2, to help evaluate and assess relationships between samples, which might help identify potential outliers early on. Overall the sequencing quality appears reasonable, as most reads have a high Phred quality score and STAR successfully maps reads in several samples. Most reads also appear to pass trimming filers. However STAR mapping rates do vary quite a lot, from 28.2% to 94.6%. Espacially sample SRX19144492 has a poor mapping, followed by SX19144491 at 59.5% and SRX19144489 at 65.2%. These samples should be investigated further, or should be excluded from further analysis.  
Ribosomal rRNA, is responsible for protein synthesis, and as such is present within the cell in a rather high proportion. Therefore rRNA can reduce the sequencing depth available for transcripts of interest, which can thereby reduce the sensitivity for lowely expressed genes. In the multiqc report this can be checked within the Biotype Counts table, which shows reads overlapping genomic features of differenct biotypes. Most samples have a low percentage of rRNA, but the Samples SRX144481(24.4%), SRX19144484 (9.3%), SRX19144489 (21.3%), stand out with a rather high rRNA percentage. So in these samples rRNA content may reduce the sequencing sensitivity and depth in certain samples. Duplication reads can for example arise from PCR amplification or high biological expression of particular genes. Consequently high duplication does not necessarily indicate poor data quality. In this report the duplication rate varies from 23.1% to 88.0% (Samples with high number of duplication: SRX19144491, SRX19144486, SRX19144489, SRX19144493). These values should be considered carefully, to ensure that a high technical number of duplications, does not infer with the data quality, as a high number of of duplications, with the same number of genes, means with have less genes covered in total. 
The GC content represents the amount of guanine and cytosine bases found within the reads. FastQC evaluetes the GC distribution to identify possible biases, as a high GC content can indicate contamiantion. In the multiqc report raw-read GC content ranges from 47% to 63%. Espacially sample SRX19144492 has a particularly high GC content, which is suspicious since this sample also has a rather low mapping rate. 
Poor mapping rates can arise due to a number of things. For example degraded RNA, poor sequencing quality, adapter contamination, excessive trimming, high rRNA content, insufficant library depth, and contamination. In this dataset most important issues are low STAR mapping rates in certains samples, high duplication rates, and high GC content.*

Would you exclude any samples? If yes, which and why?

*The sample SRX19144492, this sample has a very high GC amount. It also has a very low percentage of aligned and mapped reads.*

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

*To continue the experiment, we would know be interested in how the expression of Genes differs between the experimental groups, so the next step would be differential expression analysis.*

*Python package for differential analysises, maybe also pca: scikit-learn.
Matplotlib for vulcano plots and different plots. Numpy for calculations and normaliation. (Maybe PyDESeq2, or something from Biopython).*